In [151]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# =============== PARSER FUNCTION ===============

def parse_screener_results(html: str):
    """Parse one Screener results page into a DataFrame."""
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for block in soup.select('div.flex-row.flex-space-between.flex-align-center'):
        company_tag = block.select_one("a[href*='/company/']")
        if not company_tag:
            continue

        company_name = company_tag.get_text(strip=True)
        company_link = "https://www.screener.in" + company_tag["href"]

        # Price / MCap / PE
        price, mcap, pe = None, None, None
        for sub in block.select("span.sub"):
            text = sub.get_text(" ", strip=True)
            if "Price" in text:
                price = sub.select_one(".strong").get_text(strip=True)
            elif "M.Cap" in text:
                mcap = sub.select_one(".strong").get_text(strip=True)
            elif "PE" in text:
                pe = sub.select_one(".strong").get_text(strip=True)

        # PDF link if available
        pdf_tag = block.select_one("a[href*='/company/source/quarter/']")
        pdf_link = "https://www.screener.in" + pdf_tag["href"] if pdf_tag else None

        # Financial table
        table_div = block.find_next("div", class_="bg-base")
        if not table_div:
            continue

        table = table_div.select_one("table")
        if not table:
            continue

        headers = [th.get_text(strip=True) for th in table.select("thead th")]

        for row in table.select("tbody tr"):
            tds = row.select("td")
            if len(tds) < 2:
                continue

            metric = tds[0].get_text(" ", strip=True)

            # --- Extract YOY value (+/-) properly ---
            yoy_tag = tds[1]
            yoy_html = str(yoy_tag)

            # Try to extract directly from the <span class="change ..."> tag
            change_span = yoy_tag.select_one("span.change.up, span.change.down")

            if change_span:
                yoy_text = change_span.get_text(" ", strip=True)
                yoy_html = str(change_span)
            else:
                # fallback if the structure is unexpected
                yoy_text = yoy_tag.get_text(" ", strip=True)

            # Determine + or - sign based on class
            if 'change up' in yoy_html:
                sign = '+'
            elif 'change down' in yoy_html:
                sign = '-'
            else:
                sign = ''

            # Extract the numeric % value (ignore any tooltip text)
            matches = re.findall(r'(\d+(?:\.\d+)?)\s*%', yoy_text)
            if matches:
                yoy = sign + matches[-1]
            else:
                yoy = None

           # Dynamically map quarter values
            values = {}
            for i in range(2, len(headers)):
                col = headers[i]
                if i < len(tds):
                    val_tag = tds[i]
                    # Extract text from td
                    val_text = val_tag.get_text(" ", strip=True) if hasattr(val_tag, 'get_text') else str(val_tag)
                    # Optional: clean commas, ₹, brackets
                    val_clean = re.sub(r"[₹,\[\]]", "", val_text).strip()
                    # Convert to float if possible
                    try:
                        val_num = float(val_clean)
                    except:
                        val_num = val_clean if val_clean else None
                    values[col] = val_num
                else:
                    values[col] = None

            records.append({
                "Company": company_name,
                "Company URL": company_link,
                "PDF": pdf_link,
                "Price": price,
                "MCap": mcap,
                "PE": pe,
                "Metric": metric,
                "YOY": yoy,
                **values
            })


    return pd.DataFrame(records)
    

# =============== FETCH FUNCTION ===============

def fetch_screener_page(sessionid, csrftoken=None, page=1, day=None, month=None, year=None):
    """Fetch one Screener results page with cookies."""
    url = "https://www.screener.in/results/latest/"
    cookies = {"sessionid": sessionid}
    if csrftoken:
        cookies["csrftoken"] = csrftoken

    params = {
        "p": page,
        "result_update_date__day": day,
        "result_update_date__month": month,
        "result_update_date__year": year,
    }

    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, params=params, cookies=cookies, headers=headers)

    if "Login" in res.text or "Register" in res.text:
        raise Exception("❌ Session expired or invalid cookies.")

    return res.text

# =============== MAIN LOOP ===============

def scrape_all_results(sessionid, csrftoken=None, day=7, month=11, year=2025, max_pages=5):
    """Loop through multiple pages and append results into one DataFrame."""
    master_df = pd.DataFrame()

    for page in range(1, max_pages + 1):
        print(f"Fetching page {page}...")
        html = fetch_screener_page(sessionid, csrftoken, page, day, month, year)
        df = parse_screener_results(html)

        if df.empty:
            print("No data found, stopping.")
            break

        master_df = pd.concat([master_df, df], ignore_index=True)
        print(f"✅ Added {len(df)} rows (Total: {len(master_df)})")

        time.sleep(1)  # polite delay

    return master_df

In [ ]:
# =============== RUN ===============

SESSIONID = "f8vcxbnz9wth6llp7h5ebj7djq7q02lv"
CSRFTOKEN = None  # optional

df_full = scrape_all_results(SESSIONID, CSRFTOKEN, day=7, month=11, year=2025, max_pages=3)

print(f"\nTotal records collected: {len(df_full)}")

# ---- Data cleanup
df_all = df_full.copy()
df_all["MCap"] = df_all["MCap"].fillna(0)
df_all.drop(columns=['Company URL', 'PDF','Mar 2025', ''], axis=1, inplace=True)
# Convert YOY to numeric with proper + / - signs
df_all["YOY"] = pd.to_numeric(df_all["YOY"], errors="coerce")
df_all.iloc[:, 6] = pd.to_numeric(df_all.iloc[:, 6], errors="coerce")
df_all.iloc[:, 7] = pd.to_numeric(df_all.iloc[:, 7], errors="coerce")
df_all.loc[:, "MCap"] = (
    df_all["MCap"].astype(str).str.replace(",", "", regex=False).astype(float)
)
# ---- Find companies with EPS ⇡ and > 20% ,Mcap > 500, PE > 10 and < 100 ----
eps_mask = (
    (df_all["Metric"].str.lower().str.strip() == "eps") &
    (df_all["YOY"] > 20) &
    (df_all["MCap"].astype(float) > 500) & 
    (df_all["PE"].astype(float) > 10) &
    (df_all["PE"].astype(float) < 100)
)

# Get all companies meeting EPS criteria
eps_companies = df_all.loc[eps_mask, "Company"].unique()
print('Companies with EPS Growth more than 20%:',len(eps_companies), eps_companies)
filtered_df = df_all[df_all["Company"].isin(eps_companies)]
filtered_df = filtered_df.reset_index(drop=True)

# Calculate QoQ using column indices
filtered_df["QoQ"] = ((filtered_df.iloc[:, 6].astype(float) - filtered_df.iloc[:, 7].astype(float)) / filtered_df.iloc[:, 7].astype(float)) * 100
filtered_df["QoQ"] = filtered_df["QoQ"].round(2)
# Find the index of the 'YOY' column
yoy_idx = filtered_df.columns.get_loc("YOY")

# Move 'QoQ' column right after 'YOY'
filtered_df.insert(yoy_idx + 1, "QoQ", filtered_df.pop("QoQ"))

display(HTML(f"""
<div style="height:500px; overflow:auto">
{filtered_df.to_html(max_rows=None, max_cols=None)}
</div>
"""))
filtered_df.to_csv('Results_07112025.csv', index=False)

Fetching page 1...
✅ Added 100 rows (Total: 100)
Fetching page 2...
✅ Added 100 rows (Total: 200)
Fetching page 3...
✅ Added 20 rows (Total: 220)

Total records collected: 220
Companies with EPS Growth more than 20%: 12 ['Deep Industries' 'Shyam Metalics' 'Aegis Logistics' "Divi's Lab."
 'Vishnu Chemicals' 'Lumax Industries' 'KPI Green Energy' 'K.P. Energy'
 'Hexaware Tech.' 'Jeena Sikho' 'Highway Infra' 'Paradeep Phosph.']


,Company,Price,MCap,PE,Metric,YOY,QoQ,Sep 2025,Jun 2025,Sep 2024
0,Deep Industries,511,3270.0,22.3,Sales,64.0,3.47,179.00,173.00,109.0
1,Deep Industries,511,3270.0,22.3,EBIDT,51.0,5.71,66.70,63.10,44.1
2,Deep Industries,511,3270.0,22.3,Net profit,69.0,7.26,50.20,46.80,29.8
3,Deep Industries,511,3270.0,22.3,EPS,69.0,7.39,7.85,7.31,4.65
4,Shyam Metalics,861,24022.0,48.6,Sales,-1.0,-4.62,1671.00,1752.00,1694.0
5,Shyam Metalics,861,24022.0,48.6,EBIDT,22.0,-18.22,211.00,258.00,173.0
6,Shyam Metalics,861,24022.0,48.6,Net profit,25.0,-19.05,136.00,168.00,108.0
7,Shyam Metalics,861,24022.0,48.6,EPS,25.0,-19.13,4.86,6.01,3.88
8,Aegis Logistics,760,26683.0,40.2,Sales,31.0,33.45,2294.00,1719.00,1750.0
9,Aegis Logistics,760,26683.0,40.2,EBIDT,30.0,21.25,291.00,240.00,224.0


In [159]:
df_full.head()

,Company,Company URL,PDF,Price,MCap,PE,Metric,YOY,Sep 2025,Jun 2025,Sep 2024,Mar 2025,
0,K.P. Energy,https://www.screener.in/company/KPEL/consolida...,https://www.screener.in/company/source/quarter...,442,"2,963",22.2,Sales,+51,301.00,219.0,199.0,NaN,NaN
1,K.P. Energy,https://www.screener.in/company/KPEL/consolida...,https://www.screener.in/company/source/quarter...,442,"2,963",22.2,EBIDT,+64,65.70,48.4,40.1,NaN,NaN
2,K.P. Energy,https://www.screener.in/company/KPEL/consolida...,https://www.screener.in/company/source/quarter...,442,"2,963",22.2,Net profit,+44,35.90,25.4,24.9,NaN,NaN
3,K.P. Energy,https://www.screener.in/company/KPEL/consolida...,https://www.screener.in/company/source/quarter...,442,"2,963",22.2,EPS,+44,5.37,3.8,3.74,NaN,NaN
4,Hexaware Tech.,https://www.screener.in/company/HEXT/consolida...,https://www.screener.in/company/source/quarter...,674,"41,078",30.9,Sales,+11,3484.00,3261.0,3136.0,NaN,NaN


,Company,Price,MCap,PE,Metric,YOY,QoQ,Sep 2025,Jun 2025,Sep 2024
0,Amara Raja Ener.,980,17944.0,22.3,Sales,8.0,1.13,3388.0,3350.0,3136.0
1,Amara Raja Ener.,980,17944.0,22.3,EBIDT,-8.0,4.91,406.0,387.0,441.0
2,Amara Raja Ener.,980,17944.0,22.3,Net profit,-12.0,55.67,302.0,194.0,241.0
3,Amara Raja Ener.,980,17944.0,22.3,EPS,26.0,55.85,16.52,10.6,13.15
4,Alivus Life,914,11209.0,21.2,Sales,16.0,-2.33,588.0,602.0,507.0
5,Alivus Life,914,11209.0,21.2,EBIDT,33.0,4.07,179.0,172.0,134.0
6,Alivus Life,914,11209.0,21.2,Net profit,36.0,6.56,130.0,122.0,95.3
7,Alivus Life,914,11209.0,21.2,EPS,36.0,6.96,10.6,9.91,7.78
8,Dam Capital Advi,269,1903.0,17.0,Sales,69.0,247.40,107.0,30.8,63.3
9,Dam Capital Advi,269,1903.0,17.0,EBIDT,124.0,1177.78,75.9,5.94,33.9
